In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/raghunathangrdr/zoo-data-dataset/zoo_data-1.csv')
df

In [ ]:
df.tail(10)

In [ ]:
#columns info
df.info()

Column Definitions
* animal_name: The unique name of the animal (e.g., lion, penguin, frog).
* hair: Indicates if the animal has hair or fur (1 = Yes, 0 = No).
* feathers: Indicates if the animal has feathers (1 = Yes, 0 = No).
* eggs: Indicates if the animal lays eggs (1 = Yes, 0 = No).
* milk: Indicates if the animal produces milk/is a mammal (1 = Yes, 0 = No).
* airborne: Indicates if the animal can fly (1 = Yes, 0 = No).
* aquatic: Indicates if the animal lives in or near water (1 = Yes, 0 = No).
* predator: Indicates if the animal is a predator (1 = Yes, 0 = No).
* toothed: Indicates if the animal has teeth (1 = Yes, 0 = No).
* backbone: Indicates if the animal has a backbone/vertebral column (1 = Yes, 0 = No).
* breathes: Indicates if the animal breathes air (1 = Yes, 0 = No).
* venomous: Indicates if the animal is venomous or poisonous (1 = Yes, 0 = No).
* fins: Indicates if the animal has fins (1 = Yes, 0 = No).
* legs: The number of legs the animal has (0, 2, 4, 5, 6, or 8).
* tail: Indicates if the animal has a tail (1 = Yes, 0 = No).
* domestic: Indicates if the animal is domesticated (1 = Yes, 0 = No).
* catsize: Indicates if the animal is relatively large (roughly the size of a house cat or larger) (1 = Yes, 0 = No).

In [ ]:
#print number of missing values
print(df.isnull().sum())

In [ ]:
df.describe().T

In [ ]:
from sklearn.preprocessing import StandardScaler
features = ['hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic',
            'predator', 'toothed', 'backbone', 'breathes', 'venomous',
            'fins', 'legs', 'tail', 'domestic', 'catsize']

# Standardisation of feature
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

print(df_scaled.head())

**Since we biologically know there are roughly 7 main animal classes (Mammal, Bird, Reptile, Fish, Amphibian, Bug, Invertebrate), we will tell the K-Means algorithm to find 7 clusters.**

In [ ]:
from sklearn.cluster import KMeans

# 1. Isolate the features (Drop the name so the model only looks at traits)
X = df.drop('animal_name', axis=1)

# 2. Initialize K-Means (We guess 7 clusters based on biological taxonomy)
kmeans = KMeans(n_clusters=7, random_state=42, n_init=10)

# 3. Fit the model and predict the clusters
cluster_labels = kmeans.fit_predict(X)

# 4. Add the predicted clusters back to our original dataframe so we can see them
df['Predicted_Cluster'] = cluster_labels

print("Clustering complete! Added 'Predicted_Cluster' to the dataframe.")

In [ ]:
#animals placed in Cluster 0
print("Animals in Cluster 0:")
print(df[df['Predicted_Cluster']==0]['animal_name'].values)
print("\n")

#animals placed in Cluster 1
print("Animals in Cluster 1:")
print(df[df['Predicted_Cluster']==1]['animal_name'].values)

We can check further for the next 5 clusters remaining as well

**Because our data has 16 columns(dimensions), we can't plot it on a standard 2D graph. We need to use PCA to mathematically squash those 16 columns down into 2 columns to make a scatter plot.**

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

pca=PCA(n_components=2)
pca_features=pca.fit_transform(X)

plot_df = pd.DataFrame(data=pca_features, columns=['Component_1', 'Component_2'])
plot_df['Cluster'] = cluster_labels
plot_df['Name'] = df['animal_name']

plt.figure(figsize=(10, 8))
sns.scatterplot(x='Component_1',y='Component_2',hue='Cluster', 
                data=plot_df,palette='viridis',s=100)

plt.title('Animal Clusters Visualized using PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Predicted Cluster')
plt.show()

We evaluate clustering models by checking how similar an animal is to its own cluster compared to other clusters. We use the Silhouette Score (ranges from -1 to 1; higher is better).

In [ ]:
from sklearn.metrics import silhouette_score

sil_score=silhouette_score(X, cluster_labels)
print(f"K-Means Silhouette Score (7 clusters): {sil_score:.3f}")

# Optional: Test different numbers of clusters to see if the data naturally 
# groups into a different number (e.g., maybe 5 groups instead of 7)
scores = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    scores.append(silhouette_score(X, labels))

plt.figure(figsize=(8, 4))
plt.plot(range(2, 10), scores, marker='o', linestyle='--')
plt.title('Silhouette Score vs. Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.show()

Applying hierarchial clustering algorithm

In [ ]:
import scipy.cluster.hierarchy as sch

plt.figure(figsize=(15,8))
plt.title("Animal Taxonomy Dendrogram")

# Create a dendrogram using Ward's method
dendrogram =sch.dendrogram(sch.linkage(X,method='ward'),labels=df['animal_name'].values, leaf_rotation=90, leaf_font_size=10)

plt.xlabel("Animals")
plt.ylabel("Euclidean Distances")
plt.tight_layout()
plt.show()

In [ ]:
# Group by the predicted cluster and calculate the average for each trait
# We use numeric_only=True to ignore the 'animal_name' string column
cluster_profiles = df.groupby('Predicted_Cluster').mean(numeric_only=True)

print("Average traits for Cluster 0:")
print(cluster_profiles.loc[0])

plt.figure(figsize=(12, 6))
sns.heatmap(cluster_profiles, annot=True, cmap='YlGnBu', fmt=".2f")
plt.title('Average Traits per Cluster')
plt.ylabel('Cluster Number')
plt.xlabel('Animal Traits')
plt.show()

How to read it- Eg: If a cluster has a 1.0 for milk and a 1.0 for hair,  the algorithm successfully reinvented the concept of "Mammals" entirely on its own. If a cluster has high values for aquatic and fins, it found the "Fish".